# Simulasi NILM: Pengenalan Peralatan via Electrical Fingerprint + KNN

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hmdilham/brainstorming-project/blob/nilm/NILM%20Project/simulation/NILM_KNN_Simulation.ipynb)

Simulasi *end-to-end* dari arsitektur **Tier 1 — KNN Langsung** pada [README proyek](../README.md):
tiga peralatan rumah tangga dikenali dari **sidik jari kelistrikan** 5 dimensi
`Φ = [P, Q, Inrush_ratio, H3, H5]`, dengan harmonik diekstraksi menggunakan **algoritma Goertzel**
pada sinyal 860 SPS (meniru ADS1115).

| Perangkat | Karakter fisik | Ciri fingerprint |
|---|---|---|
| **Kipas angin** (~60 W) | motor induksi kecil | Q signifikan, inrush ~2–3× |
| **Pompa air** (~350 W) | motor induksi besar | P & Q besar, inrush 5–7× |
| **Lampu LED 50 W** (driver murah) | beban switching | H3 ~9%, H5 ~21%, Q kecil kapasitif |

Alur simulasi = alur sistem nyata di ESP32:

```
v(t), i(t) @ 860 SPS ──► Goertzel @ 50/150/250 Hz ──► Φ = [P, Q, Inrush, H3, H5]
                                                          │
   onboarding 5-shot ──► support set ("Flash")  ◄─────────┘
                                                          │
   event baru ──► KNN (K=3, Euclidean, majority vote) ──► label perangkat
```

> Semua sinyal di sini **sintetis namun berbasis fisika** (deret Fourier arus + selubung
> transien eksponensial + noise sensor), karena dataset hardware belum tersedia.
> Notebook ini hanya butuh `numpy` & `matplotlib` — langsung jalan di Google Colab.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

FS = 860.0          # sample rate ADS1115 (sampel/detik)
F0 = 50.0           # frekuensi fundamental grid Indonesia (Hz)
VRMS = 220.0        # tegangan nominal (V)
N_GOERTZEL = 430    # jendela analisis 0.5 s = 25 siklus penuh 50 Hz
                    # bin k = f*N/FS -> 50 Hz: 25, 150 Hz: 75, 250 Hz: 125
                    # semuanya bulat -> bebas spectral leakage
K_SHOT = 5          # contoh onboarding per perangkat (few-shot)
K_NN = 3            # tetangga terdekat untuk majority vote
RNG = np.random.default_rng(42)

# palet kategorikal (tervalidasi aman untuk buta warna)
COLORS = {"Kipas Angin": "#2a78d6", "Pompa Air": "#1baf7a", "Lampu LED 50W": "#eda100"}
INK, MUTED = "#333331", "#77776c"

plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "font.size": 10, "figure.facecolor": "white",
})

## 1. Algoritma Goertzel

Goertzel adalah *resonator* IIR orde-2 yang menghitung **satu bin DFT** dengan biaya
O(N) per frekuensi — jauh lebih ringan dari FFT penuh bila hanya butuh 3 frekuensi
(50, 150, 250 Hz). Inilah alasan ia dipilih untuk ESP32 di README.

Rekurensi per sampel (hanya 1 perkalian + 2 penjumlahan):

$$s[n] = x[n] + 2\cos(\omega_k)\,s[n-1] - s[n-2]$$

dan di akhir jendela, komponen real/imajiner diperoleh dari dua state terakhir.

In [ ]:
def goertzel(x, f_target, fs):
    """Kembalikan (amplitudo, fase) komponen f_target dalam sinyal x."""
    n = len(x)
    k = f_target * n / fs                 # pilihan N membuat k bulat
    w = 2.0 * np.pi * k / n
    cw, sw = np.cos(w), np.sin(w)
    coeff = 2.0 * cw

    s_prev = s_prev2 = 0.0
    for sample in x:                      # loop per-sampel, persis seperti di MCU
        s = sample + coeff * s_prev - s_prev2
        s_prev2, s_prev = s_prev, s

    real = s_prev - s_prev2 * cw
    imag = s_prev2 * sw
    mag = 2.0 / n * np.hypot(real, imag)  # skala -> amplitudo puncak
    return mag, np.arctan2(imag, real)

# --- sanity check: sinus murni 50 Hz amplitudo 1.0 harus terbaca 1.0,
#     dan bin 150/250 Hz harus ~0 (tidak ada kebocoran antar-bin)
tt = np.arange(N_GOERTZEL) / FS
pure = np.sin(2 * np.pi * F0 * tt)
for f in (50, 150, 250):
    m, _ = goertzel(pure, f, FS)
    print(f"  bin {f:>3.0f} Hz -> amplitudo terukur {m:.6f}")

## 2. Model Fisika Tiga Perangkat

Arus tiap perangkat dibangkitkan sebagai deret Fourier (fundamental + H3 + H5) dengan
sudut fase sesuai *power factor*, lalu dikalikan selubung transien eksponensial untuk
meniru *inrush current*:

$$i(t) = \underbrace{I_1\big[\sin(\omega t - \varphi) + h_3\sin(3\omega t + \ldots) + h_5\sin(5\omega t + \ldots)\big]}_{\text{steady state}} \times \underbrace{\big[1 + (r_{inrush} - 1)e^{-t/\tau}\big]}_{\text{selubung transien}}$$

Setiap event diberi *jitter* parameter (±4–12%) dan noise sensor, meniru variasi
tegangan PLN, suhu motor, dan toleransi komponen — sesuai catatan README bahwa
onboarding perlu 3–5 contoh pada kondisi berbeda.

In [ ]:
APPLIANCES = {
    "Kipas Angin": dict(
        P=60.0, pf=0.72, lagging=True,     # motor induksi: Q signifikan
        h3=0.045, h5=0.020,                # hampir linear, harmonik kecil
        inrush=2.5, tau=0.12,
    ),
    "Pompa Air": dict(
        P=350.0, pf=0.68, lagging=True,    # motor induksi besar
        h3=0.055, h5=0.025,
        inrush=6.0, tau=0.20,              # ciri khas: inrush 5-7x
    ),
    "Lampu LED 50W": dict(
        P=50.0, pf=0.90, lagging=False,    # driver switching, sedikit kapasitif
        h3=0.09, h5=0.21,                  # profil README: H3 9%, H5 21%
        inrush=1.3, tau=0.01,
    ),
}

def synth_event(spec, rng):
    """Bangkitkan v(t), i(t) satu event penyalaan @ 860 SPS selama 1.2 s."""
    t = np.arange(int(1.2 * FS)) / FS

    P   = spec["P"] * rng.normal(1.0, 0.04)
    pf  = np.clip(spec["pf"] * rng.normal(1.0, 0.02), 0.05, 0.999)
    h3  = spec["h3"] * rng.normal(1.0, 0.10)
    h5  = spec["h5"] * rng.normal(1.0, 0.10)
    irr = spec["inrush"] * rng.normal(1.0, 0.12)
    vamp = np.sqrt(2) * VRMS * rng.normal(1.0, 0.01)

    phi = np.arccos(pf) * (1 if spec["lagging"] else -1)
    i1 = np.sqrt(2) * P / (VRMS * pf)

    w = 2 * np.pi * F0
    v = vamp * np.sin(w * t)
    i = i1 * (np.sin(w * t - phi)
              + h3 * np.sin(3 * w * t - 3 * phi + 0.5)
              + h5 * np.sin(5 * w * t - 5 * phi + 1.0))
    i *= 1.0 + (irr - 1.0) * np.exp(-t / spec["tau"])   # selubung inrush

    i += rng.normal(0, 0.008 * i1, len(t))              # noise sensor arus
    v += rng.normal(0, 0.002 * vamp, len(t))            # noise sensor tegangan
    return v, i

In [ ]:
# --- visual: transien penyalaan tiap perangkat (perhatikan skala arus & inrush)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, (name, spec) in zip(axes, APPLIANCES.items()):
    v, i = synth_event(spec, np.random.default_rng(7))
    t = np.arange(len(i)) / FS
    ax.plot(t, i, color=COLORS[name], linewidth=0.7)
    pk = np.argmax(np.abs(i[: int(0.3 * FS)]))
    ax.annotate(f"puncak {abs(i[pk]):.1f} A", xy=(t[pk], i[pk]),
                xytext=(t[pk] + 0.25, i[pk] * 0.85), color=INK, fontsize=9,
                arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.8))
    ax.set_title(name, color=INK, fontsize=11)
    ax.set_xlabel("waktu (s)")
axes[0].set_ylabel("arus (A)")
fig.suptitle("Transien penyalaan — pompa air menunjukkan inrush ~6x", y=1.03, color=INK)
plt.tight_layout(); plt.show()

# --- visual: bentuk gelombang steady-state (dinormalisasi) -> distorsi harmonik LED
fig, ax = plt.subplots(figsize=(8, 3.4))
n2 = int(2 * FS / F0)                       # 2 siklus terakhir
label_y = {"Kipas Angin": -0.78, "Pompa Air": -0.97, "Lampu LED 50W": 0.30}
for name, spec in APPLIANCES.items():
    _, i = synth_event(spec, np.random.default_rng(7))
    seg = i[-n2:] / np.max(np.abs(i[-n2:]))    # per-unit agar bentuk terbanding
    tms = np.arange(n2) / FS * 1000
    ax.plot(tms, seg, color=COLORS[name], linewidth=2, label=name)
    ax.annotate(name, xy=(tms[-1] + 0.6, label_y[name]),   # label diposisikan manual
                color=COLORS[name], fontsize=9, va="center")  # (kipas & pompa berimpit)
ax.set_xlim(0, 48); ax.set_xlabel("waktu (ms)"); ax.set_ylabel("arus (per-unit)")
ax.set_title("Steady state: gelombang LED terdistorsi (H3+H5), motor hampir sinusoidal", color=INK)
ax.legend(frameon=False, loc="lower left", fontsize=8)
plt.tight_layout(); plt.show()

## 3. Ekstraksi Fitur `Φ = [P, Q, Inrush_ratio, H3, H5]`

Semua fitur diturunkan dari **empat panggilan Goertzel** + aritmetika dasar — tanpa FFT:

- **P, Q** — dari fasor fundamental tegangan & arus: $P = \tfrac{1}{2}V_1 I_1\cos\Delta\varphi$, $Q = \tfrac{1}{2}V_1 I_1\sin\Delta\varphi$
- **Inrush_ratio** — puncak arus 0.3 s pertama ÷ puncak steady-state
- **H3, H5** — amplitudo Goertzel di 150/250 Hz sebagai persen terhadap fundamental

In [ ]:
FEAT_NAMES = ["P (W)", "Q (var)", "Inrush", "H3 (%)", "H5 (%)"]

def extract_features(v, i):
    vs, is_ = v[-N_GOERTZEL:], i[-N_GOERTZEL:]      # 25 siklus steady-state terakhir

    v1, phv  = goertzel(vs,  F0,     FS)
    i1, phi1 = goertzel(is_, F0,     FS)
    i3, _    = goertzel(is_, 3 * F0, FS)            # H3 @ 150 Hz
    i5, _    = goertzel(is_, 5 * F0, FS)            # H5 @ 250 Hz

    dphi = phv - phi1
    P = 0.5 * v1 * i1 * np.cos(dphi)
    Q = 0.5 * v1 * i1 * np.sin(dphi)

    peak_transient = np.max(np.abs(i[: int(0.3 * FS)]))
    peak_steady = np.sqrt(2) * np.sqrt(np.mean(is_**2))

    return np.array([P, Q, peak_transient / peak_steady, 100 * i3 / i1, 100 * i5 / i1])

# contoh: fingerprint satu event pompa air
v, i = synth_event(APPLIANCES["Pompa Air"], np.random.default_rng(1))
print("Φ pompa air:", np.round(extract_features(v, i), 2))

## 4. Fase Onboarding (Few-Shot, K=5)

Meniru alur pengguna di README: nyalakan perangkat → sistem merekam fingerprint →
ulangi 5×. Vektor `Φ` disimpan apa adanya ke *support set* — **ukuran model 0 KB,
tanpa training, tanpa gradient descent**.

In [ ]:
names = list(APPLIANCES)
Xs, ys = [], []
for name in names:
    print(f"\n  {name}")
    print("    shot |" + "|".join(f"{f:>9}" for f in FEAT_NAMES))
    for k in range(K_SHOT):
        phi = extract_features(*synth_event(APPLIANCES[name], RNG))
        Xs.append(phi); ys.append(name)
        print(f"      #{k+1} |" + "|".join(f"{x:9.2f}" for x in phi))
Xs, ys = np.array(Xs), np.array(ys)

print("\n  Fingerprint rata-rata:")
print(f"  {'Perangkat':<16}|" + "|".join(f"{f:>9}" for f in FEAT_NAMES))
for name in names:
    m = Xs[ys == name].mean(axis=0)
    print(f"  {name:<16}|" + "|".join(f"{x:9.2f}" for x in m))

In [ ]:
# --- visual: ruang fitur -- dua proyeksi 2D dari fingerprint 5 dimensi
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
# offset label per panel agar tidak bertabrakan (kipas & pompa bertetangga di H3-H5)
offsets = {("Kipas Angin", 1): (0, 14), ("Pompa Air", 1): (0, 14), ("Lampu LED 50W", 1): (70, -4),
           ("Kipas Angin", 2): (-10, 16), ("Pompa Air", 2): (14, 16), ("Lampu LED 50W", 2): (-60, -14)}
for name in names:
    pts = Xs[ys == name]
    for panel, (ax, ix, iy) in enumerate(((ax1, 0, 1), (ax2, 3, 4)), start=1):
        ax.scatter(pts[:, ix], pts[:, iy], s=55, color=COLORS[name],
                   edgecolors="white", linewidths=1.2, label=name, zorder=3)
        cx, cy = pts[:, ix].mean(), pts[:, iy].mean()
        ax.annotate(name, xy=(cx, cy), xytext=offsets[(name, panel)],
                    textcoords="offset points", ha="center",
                    color=COLORS[name], fontsize=9, fontweight="bold")
ax1.set_xlabel("P — daya aktif (W)"); ax1.set_ylabel("Q — daya reaktif (var)")
ax1.set_title("Motor (Q > 0) vs LED (Q < 0)", color=INK, fontsize=10)
ax2.set_xlabel("H3 (%)"); ax2.set_ylabel("H5 (%)")
ax2.set_title("Harmonik: LED terpisah jauh dari motor", color=INK, fontsize=10)
ax1.legend(frameon=False, fontsize=8)
fig.suptitle("Support set hasil onboarding pada ruang fitur", y=1.02, color=INK)
plt.tight_layout(); plt.show()

## 5. Inferensi: KNN Tier-1 (K=3, Euclidean, Majority Vote)

Fitur di-*z-score* memakai statistik support set (agar P yang ratusan watt tidak
menenggelamkan H3/H5 yang berskala persen), lalu tiap event baru dicari 3 tetangga
terdekatnya. Diuji dengan **30 event baru per perangkat**.

In [ ]:
class KNNTier1:
    """Support set langsung di 'Flash' -- tanpa model ML."""
    def fit(self, X, y):
        self.mu, self.sd = X.mean(axis=0), X.std(axis=0) + 1e-9
        self.X, self.y = (X - self.mu) / self.sd, np.asarray(y)
    def predict(self, x):
        d = np.linalg.norm(self.X - (x - self.mu) / self.sd, axis=1)
        nn = np.argsort(d)[:K_NN]
        lab, cnt = np.unique(self.y[nn], return_counts=True)
        return lab[np.argmax(cnt)]

knn = KNNTier1()
knn.fit(Xs, ys)

N_TEST = 30
conf = np.zeros((3, 3), dtype=int)
test_pts = {n: [] for n in names}
for a, name in enumerate(names):
    for _ in range(N_TEST):
        phi = extract_features(*synth_event(APPLIANCES[name], RNG))
        test_pts[name].append(phi)
        conf[a, names.index(knn.predict(phi))] += 1

acc = np.trace(conf) / conf.sum() * 100
print(f"AKURASI: {acc:.1f}%  ({np.trace(conf)}/{conf.sum()} event benar)")

In [ ]:
# --- visual: confusion matrix (ramp sekuensial biru, satu hue)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list("seq_blue", ["#f5f9ff", "#cde2fb", "#3987e5", "#0d366b"])

fig, ax = plt.subplots(figsize=(4.6, 4))
ax.imshow(conf, cmap=cmap, vmin=0, vmax=N_TEST)
short = [n.split()[0] for n in names]
ax.set_xticks(range(3), short); ax.set_yticks(range(3), short)
ax.set_xlabel("prediksi"); ax.set_ylabel("aktual"); ax.grid(False)
for r in range(3):
    for c in range(3):
        ax.text(c, r, conf[r, c], ha="center", va="center", fontsize=12,
                color="white" if conf[r, c] > N_TEST * 0.6 else INK)
ax.set_title(f"Confusion matrix — akurasi {acc:.0f}%", color=INK, fontsize=11)
plt.tight_layout(); plt.show()

# separabilitas antar-centroid di ruang z-score
Z = (Xs - knn.mu) / knn.sd
cents = {n: Z[ys == n].mean(axis=0) for n in names}
print("Jarak antar-centroid (ruang z-score):")
for a in range(3):
    for b in range(a + 1, 3):
        print(f"  {names[a]:<14} <-> {names[b]:<14}: "
              f"{np.linalg.norm(cents[names[a]] - cents[names[b]]):.2f}")
intra = np.mean([np.linalg.norm(Z[ys == n] - cents[n], axis=1).mean() for n in names])
print(f"  sebaran intra-kelas rata-rata: {intra:.2f}")

## 6. Kesimpulan

Ketiga perangkat **terpisah sempurna** di ruang fitur 5-dimensi — jarak antar-centroid
~10× lebih besar dari sebaran intra-kelas — sehingga KNN K=3 mencapai akurasi 100%
pada 90 event uji. Tiap dimensi fitur menjalankan perannya sesuai desain README:

| Fitur | Peran dalam pemisahan |
|---|---|
| **P** | memisahkan pompa air (~350 W) dari dua lainnya (~50–60 W) |
| **Q** | memisahkan motor induksi (Q ≫ 0) dari LED (Q ≲ 0) — kipas vs lampu |
| **Inrush** | konfirmasi motor besar (6×) vs kecil (2.5×) vs elektronik (1.3×) |
| **H3, H5** | menandai beban switching: LED (9%, 21%) vs motor (<6%, <3%) |

**Catatan jujur untuk laporan penelitian:** ini adalah *upper bound* — sinyal sintetis
dengan noise Gaussian sederhana. Akurasi nyata (estimasi README: 75–82% untuk Tier 1)
akan turun karena noise sensor Hall non-Gaussian, fluktuasi grid, kondisi multi-beban,
dan perangkat dengan fingerprint berdekatan (mis. dua motor kecil). Langkah berikutnya:
validasi dengan data ADS1115 riil, lalu bandingkan Tier 1 vs Tier 2 (MLP encoder + KNN).